In [ ]:
# ============================================================
# SignLingo Round 2 -- Phase 1: prepare_features
# ============================================================
# Turns the compact 4-block .npy tree written by the extraction repo into one
# clean, de-duplicated, class-balanced dataset.npz that every later notebook
# reads. No training happens here and nothing is augmented -- augmentation is
# train-only and belongs inside each fold (Next-Steps 5.4).
#
# Input :  features_compact/<class>/<file>.npy, each (30, 146)
# Output:  dataset.npz  ->  X (N,30,146) float32, y (N,) int, groups (N,) str,
#                           class_names, arm_cuts
#
# Order inside this notebook:
#   1 load + assign signers      4 balance + coverage matrix
#   2 de-duplicate               5 separability gate  <-- read this number
#   3 mirror + audit             6 save
import os
import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.family": "DejaVu Sans"})

In [ ]:
# ============================================================
# 0 -- Config
# ============================================================
DATA_DIR        = "features_compact"   # drop the extraction repo's output here
OUT_PATH        = "dataset.npz"
SEQUENCE_LENGTH = 30
DIM_COMPACT     = 146
MAX_SAMPLES     = 110                  # per class, after de-duplication
RANDOM_STATE    = 42

# Block layout, matching the extraction repo's compact_features.py. Each arm is
# a prefix of the vector, so an ablation arm is a plain column slice.
#   A hands 126 | B articulator pose 13 | C postural NMS 3 | D facial NMS 4
#
# Block B is limb DIRECTIONS (upper-arm and forearm unit vectors, both sides) plus
# one shoulder-tilt angle, not the 18 raw coordinates of pose 11..16. The earlier
# coordinate form raised linear signer-recoverability from 65.5% to 94.6% on its
# own, because elbow and wrist positions encode arm length. Unit vectors keep the
# motion and drop the length. Wrist position is not lost: hand landmark 0 is the
# wrist and already sits in block A.
ARM_CUTS = {"A_hands": 126, "AB_artic": 139, "ABC_postural": 142, "ABCD_facial": 146}

# Block D is 4 features, not the 5 in Next-Steps 4.6. Eye openness is not
# computable: the retained face-landmark subset has lips, eyebrows and cheeks
# but no eyelid contour. Recorded here so the paper's table matches the code.
BLOCK_D_FEATURES = ["mouth_aperture", "mouth_width", "eyebrow_raise_L", "eyebrow_raise_R"]

# Near-duplicate cut-off, as a fraction of sequence magnitude. Phase 0 used 0.05
# on 447 dimensions; distance distributions differ at 146, so cell 2 plots the
# histogram and prints the spread. Set this from that plot, do not assume.
NEAR_DUP_REL_THRESHOLD = 0.05

# The 20 NMS-focused classes, for the 5.3 breakdown. UNVERIFIED RECONSTRUCTION,
# carried over unchanged so the split stays pre-registered rather than re-derived
# after seeing Round 2 numbers.
NMS_FOCUSED_CLASSES = {
    "Apa", "Bagaimana", "Berapa", "Dimana", "Kapan", "Kemana", "Siapa",
    "Bingung", "Marah", "Ramah", "Sabar", "Sedih", "Senang", "Baik",
    "Apa Kabar", "Halo", "Terima Kasih", "Tinggi", "Pendek", "Melihat",
}


def classify_signer(fname):
    """Signer identity is recorded nowhere, so it is recovered from the four
    filename conventions. Confirmed against per-class counts in Phase 0."""
    stem = re.sub(r"\s*\(\d+\)$", "", os.path.splitext(fname)[0])
    if stem.upper().startswith("BISINDO_"):
        return "Signer_D_bisindo"
    if re.fullmatch(r"\d+", stem):
        return "Signer_A_numeric"
    if re.search(r"-\d+$", stem):
        return "Signer_B_dash"
    if re.search(r"_\d+$", stem):
        return "Signer_C_underscore"
    return "Signer_UNK"

In [ ]:
# ============================================================
# 1 -- Load and assign signers
# ============================================================
assert os.path.isdir(DATA_DIR), (
    DATA_DIR + "/ not found. Copy the compact feature tree from the extraction "
    "repo into this folder first."
)

class_names = sorted(d for d in os.listdir(DATA_DIR)
                     if os.path.isdir(os.path.join(DATA_DIR, d)))
num_classes = len(class_names)

# The two feature trees disagree on class-folder naming: "Apa Kabar" in the old
# 447-dim tree, "apa_kabar" in the extraction repo's compact tree. Matched on a
# normalised key so the 20/20 NMS split cannot silently select nothing.
def _key(name):
    return name.lower().replace("_", " ").strip()


_nms_keys = {_key(c) for c in NMS_FOCUSED_CLASSES}
nms_mask = np.array([_key(c) in _nms_keys for c in class_names])
_unmatched = sorted(_nms_keys - {_key(c) for c in class_names})
print("NMS-focused classes matched: %d of %d" % (nms_mask.sum(), len(_nms_keys)))
if _unmatched:
    print("  WARNING: %d NMS class name(s) match no folder: %s"
          % (len(_unmatched), _unmatched))
    print("  The 20/20 breakdown in run_ablation will be wrong. Fix the names")
    print("  in NMS_FOCUSED_CLASSES before continuing.")

X_raw, y_raw, g_raw, f_raw, skipped = [], [], [], [], []
for ci, action in enumerate(class_names):
    adir = os.path.join(DATA_DIR, action)
    for fname in sorted(f for f in os.listdir(adir) if f.endswith(".npy")):
        seq = np.load(os.path.join(adir, fname))
        if seq.shape != (SEQUENCE_LENGTH, DIM_COMPACT):
            skipped.append((action, fname, seq.shape))
            continue
        X_raw.append(seq.astype(np.float32))
        y_raw.append(ci)
        g_raw.append(classify_signer(fname))
        f_raw.append(action + "/" + fname)

X_raw = np.asarray(X_raw, dtype=np.float32)
y_raw = np.asarray(y_raw)
g_raw = np.asarray(g_raw)
f_raw = np.asarray(f_raw)

print("Loaded %d sequences, %d classes, shape %s"
      % (len(X_raw), num_classes, X_raw.shape))
if skipped:
    print("  WARNING: skipped %d file(s) with a wrong shape, e.g. %s"
          % (len(skipped), skipped[:3]))
    print("  A wrong shape usually means a stale features_compact/ from an older")
    print("  derivation. Delete it and re-derive rather than training on a mix.")
assert np.isfinite(X_raw).all(), "non-finite values in the loaded features"

signers, counts = np.unique(g_raw, return_counts=True)
print("\nSequences per signer:")
for s, c in zip(signers, counts):
    print("  %-22s %5d" % (s, c))
# Drop the unclassifiable files now rather than later. They cannot be placed in
# any LOSO fold, and leaving them in lets them consume per-class balance slots
# that should go to a known signer.
n_unk = int((g_raw == "Signer_UNK").sum())
if n_unk:
    print("  (%d unclassifiable, dropped now: they fit no fold and would take"
          % n_unk)
    print("   balance slots from a known signer)")
    known = g_raw != "Signer_UNK"
    X_raw, y_raw, g_raw, f_raw = X_raw[known], y_raw[known], g_raw[known], f_raw[known]
    signers = np.array([s for s in signers if s != "Signer_UNK"])

In [ ]:
# ============================================================
# 2 -- De-duplicate
# ============================================================
# Phase 0 found 41 exact and 50 near-duplicate pairs, all same-signer (Windows
# "(1)"/"(2)" copies). They never affected LOSO, since same-signer copies always
# land on the same side of a signer-disjoint split, but they inflate any
# sequence-level split by roughly 1pp. Removed before balancing so the per-class
# quota is filled with genuinely distinct sequences.
flat = X_raw.reshape(len(X_raw), -1)
norms = np.linalg.norm(flat, axis=1)

nn = NearestNeighbors(n_neighbors=2).fit(flat)
dist, idx = nn.kneighbors(flat)
nn_dist, nn_idx = dist[:, 1], idx[:, 1]
rel = nn_dist / np.maximum(0.5 * (norms + norms[nn_idx]), 1e-9)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(rel, bins=120, color="#4C72B0")
ax[0].axvline(NEAR_DUP_REL_THRESHOLD, color="#C44E52", ls="--",
              label="threshold %.3f" % NEAR_DUP_REL_THRESHOLD)
ax[0].set_xlabel("nearest-neighbour distance / magnitude")
ax[0].set_ylabel("sequences")
ax[0].legend()
ax[0].set_title("All sequences", fontweight="bold")
ax[1].hist(rel[rel < np.percentile(rel, 10)], bins=80, color="#4C72B0")
ax[1].axvline(NEAR_DUP_REL_THRESHOLD, color="#C44E52", ls="--")
ax[1].set_xlabel("same, lowest decile only")
ax[1].set_title("Zoom: where duplicates live", fontweight="bold")
plt.tight_layout()
plt.savefig("prep_dedup_hist.png", dpi=120)
plt.show()

print("Relative NN distance: min %.4f  p1 %.4f  p5 %.4f  median %.4f"
      % (rel.min(), np.percentile(rel, 1), np.percentile(rel, 5), np.median(rel)))
print("Threshold %.3f flags %d sequence(s). Check the zoom panel: the threshold"
      % (NEAR_DUP_REL_THRESHOLD, int((rel < NEAR_DUP_REL_THRESHOLD).sum())))
print("belongs in the gap after the duplicate spike, not inside the main bulk.")

# Keep one member of each duplicate group. Union-find over the flagged pairs, so
# a triple of copies collapses to one rather than to two.
parent = list(range(len(flat)))


def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)


for i in np.flatnonzero(rel < NEAR_DUP_REL_THRESHOLD):
    j = int(nn_idx[i])
    if y_raw[i] == y_raw[j]:            # never merge across classes
        union(int(i), j)

keep = np.array([find(i) == i for i in range(len(flat))])
print("\nExact duplicates: %d sequence(s) with a distance-0 twin"
      % int((nn_dist < 1e-6).sum()))
print("De-duplicated: %d -> %d (%d removed)"
      % (len(X_raw), int(keep.sum()), int((~keep).sum())))

cross = sum(1 for i in np.flatnonzero(rel < NEAR_DUP_REL_THRESHOLD)
            if g_raw[i] != g_raw[nn_idx[i]])
print("Of the flagged pairs, %d cross a signer boundary (Phase 0 saw none; a"
      % cross)
print("non-zero count here is worth a look before continuing).")

X, y, g, fn = X_raw[keep], y_raw[keep], g_raw[keep], f_raw[keep]

In [ ]:
# ============================================================
# 3 -- Mirror, and audit it on real data
# ============================================================
# Defined here rather than imported, so this repo needs nothing from the
# extraction repo at run time. On the compact vector every block's mirror is
# obvious and no landmark index remapping is left to get wrong (Next-Steps 4.6),
# unlike the 447-dim version, which took three attempts.
def mirror(v):
    """Left-right mirror of a compact frame or (..., 146) array."""
    m = np.asarray(v, dtype=np.float32)
    lead = m.shape[:-1]
    out = m.copy()

    hands = m[..., 0:126].reshape(lead + (2, 21, 3))[..., ::-1, :, :].copy()
    hands[..., 0] *= -1                       # swap the hands, negate x
    out[..., 0:126] = hands.reshape(lead + (126,))

    # (2 sides, 2 segments, 3): reversing the side axis swaps the arms, negating x
    # flips each direction.
    artic = m[..., 126:138].reshape(lead + (2, 2, 3))[..., ::-1, :, :].copy()
    artic[..., 0] *= -1
    out[..., 126:138] = artic.reshape(lead + (12,))

    out[..., 138] = -m[..., 138]              # shoulder tilt flips sign
    out[..., 139] = -m[..., 139]              # roll flips sign
    out[..., 140] = -m[..., 140]              # yaw flips sign
    out[..., 144] = m[..., 145]               # eyebrow raise swaps sides
    out[..., 145] = m[..., 144]
    return out                                # pitch, both mouth features: unchanged


# Audited on real sequences, not synthetic ones: the Round 1 face-pairing bug
# passed an x-correlation check at +0.99 while still pairing lips to eyebrows.
_rng = np.random.default_rng(RANDOM_STATE)
sample = X[_rng.choice(len(X), min(200, len(X)), replace=False)]
M = mirror(sample)
assert np.allclose(mirror(M), sample, atol=1e-5), "mirroring twice is not the identity"
assert np.allclose(M[..., 141], sample[..., 141]), "pitch must survive a mirror"
assert np.allclose(M[..., 142:144], sample[..., 142:144]), "mouth features must survive"
assert np.allclose(M[..., 144], sample[..., 145]), "eyebrow sides did not swap"
assert np.allclose(M[..., 0:63].reshape(-1, 21, 3),
                   sample[..., 63:126].reshape(-1, 21, 3) * [-1, 1, 1], atol=1e-5), \
    "the right hand did not land in the left hand's slot"
assert np.allclose(M[..., 126:132], sample[..., 132:138] * [-1, 1, 1, -1, 1, 1],
                   atol=1e-5), "the right arm did not land in the left arm's slot"
assert np.allclose(np.linalg.norm(sample[..., 126:138].reshape(-1, 4, 3), axis=-1),
                   1.0, atol=1e-4), "block B limb vectors are not unit length"
# The mirror is a signed permutation: every output element is exactly one input
# element times +/-1. Recovering that as two vectors lets run_ablation apply it as
# X[..., perm] * sign, so the mirror logic lives here only and cannot drift between
# notebooks. Read straight off mirror() rather than written out by hand.
_E = np.eye(DIM_COMPACT, dtype=np.float32)
_M = mirror(_E)
mirror_perm = np.argmax(np.abs(_M), axis=0)
mirror_sign = _M[mirror_perm, np.arange(DIM_COMPACT)]
assert np.allclose(np.abs(_M).sum(axis=0), 1.0), "mirror is not a signed permutation"
assert np.allclose(sample[..., mirror_perm] * mirror_sign, M, atol=1e-5), \
    "the signed-permutation form disagrees with mirror() on real data"

print("Mirror audit passed on %d real sequences:" % len(sample))
print("  double mirror = identity | hands and limb directions swap with x negated")
print("  limb vectors are unit length: block B carries direction, not arm length")
print("  mirror stored as a signed permutation, so run_ablation reuses it exactly")
print("  shoulder tilt, roll, yaw flip sign | eyebrow L/R swap | pitch, mouth same")

_f = X[0]
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5), sharex=True, sharey=True)
for a, d, t in [(ax[0], _f, "original"), (ax[1], mirror(_f), "mirrored")]:
    h = d[0, 0:126].reshape(42, 3)
    v = d[0, 126:138].reshape(4, 3)          # limb directions, drawn from the origin
    a.scatter(h[:21, 0], -h[:21, 1], s=12, label="left hand")
    a.scatter(h[21:, 0], -h[21:, 1], s=12, label="right hand")
    for j in range(4):
        a.plot([0, v[j, 0]], [0, -v[j, 1]], lw=2, alpha=.7,
               label="limb directions" if j == 0 else None)
    a.set_title(t, fontweight="bold")
    a.set_aspect("equal")
ax[0].legend(fontsize=8)
plt.suptitle("Mirror check, frame 0 of " + str(fn[0]), fontweight="bold")
plt.tight_layout()
plt.savefig("prep_mirror_audit.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# 4 -- Balance, signer-aware, and check fold coverage
# ============================================================
# Trimming to MAX_SAMPLES per class without looking at who recorded what can
# leave a class dominated by one signer. When that signer is held out, the fold
# is near zero-shot for that class and the accuracy drop is structural rather
# than a generalization failure. Filling each class round-robin across signers
# keeps the mix as even as the recordings allow.
rng = np.random.default_rng(RANDOM_STATE)
sel = []
for ci in range(num_classes):
    pool = {s: list(rng.permutation(np.flatnonzero((y == ci) & (g == s))))
            for s in np.unique(g)}
    taken = []
    while len(taken) < MAX_SAMPLES and any(pool.values()):
        for s in sorted(pool):
            if pool[s] and len(taken) < MAX_SAMPLES:
                taken.append(int(pool[s].pop()))
    sel += taken
sel = np.sort(np.asarray(sel))
X, y, g, fn = X[sel], y[sel], g[sel], fn[sel]
print("Balanced to at most %d per class: %d sequences" % (MAX_SAMPLES, len(X)))

class_counts = np.bincount(y, minlength=num_classes)
fig, ax = plt.subplots(figsize=(max(12, num_classes * 0.45), 5))
bars = ax.bar(class_names, class_counts,
              color=plt.cm.plasma(np.linspace(0.15, 0.85, num_classes)),
              edgecolor="white", linewidth=0.6)
ax.set_title("Samples per Class", fontweight="bold")
ax.set_xlabel("Sign Label")
ax.set_ylabel("Number of Sequences")
ax.tick_params(axis="x", rotation=55)
for bar, cnt in zip(bars, class_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(cnt), ha="center", va="bottom", fontsize=8, fontweight="bold")
plt.tight_layout()
plt.savefig("samples_per_class.pdf")
plt.show()
print("Min: %d  |  Max: %d  |  Mean: %.1f"
      % (class_counts.min(), class_counts.max(), class_counts.mean()))

cov = np.zeros((num_classes, len(signers)), dtype=int)
sidx = {s: i for i, s in enumerate(signers)}
for ci, sg in zip(y, g):
    cov[ci, sidx[sg]] += 1

known_cols = list(range(len(signers)))
empty = [(class_names[r], signers[c])
         for r in range(num_classes) for c in known_cols if cov[r, c] == 0]

print("\nClass x signer coverage:")
print("  %-16s" % "class" + "".join("%16s" % s.replace("Signer_", "") for s in signers))
for r in range(num_classes):
    print("  %-16s" % class_names[r] + "".join("%16d" % cov[r, c]
                                               for c in range(len(signers))))

if empty:
    print("\n  WARNING: %d class/signer cell(s) empty, e.g. %s" % (len(empty), empty[:5]))
    print("  A held-out fold for that signer is zero-shot on that class, so its")
    print("  accuracy drop is structural, not a generalization result. Fix the")
    print("  balancing or report the affected folds separately.")
else:
    print("\n  Every class has every signer. No fold is zero-shot on any class, so")
    print("  the LOSO numbers are genuine generalization measurements.")

fig, ax = plt.subplots(figsize=(6, 9))
sns.heatmap(cov, ax=ax, cmap="mako", annot=True, fmt="d", cbar=False,
            xticklabels=[s.replace("Signer_", "") for s in signers],
            yticklabels=class_names, annot_kws={"size": 7})
ax.set_title("Sequences per class and signer", fontweight="bold")
plt.tight_layout()
plt.savefig("prep_coverage.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# 5 -- Separability gate: can a linear model still tell the signers apart?
# ============================================================
# THE NUMBER THIS NOTEBOOK EXISTS FOR. Phase 0 recovered signer identity from
# pooled 447-dim landmarks at 99.7% with plain logistic regression, against a
# 44.6% majority baseline. That is the leak the compact redesign should close.
#
# Read it before booking time on the training machine. If it still says ~99%,
# the representation change did not work and the 4-arm grid is not worth its
# overnight slot. Two minutes here against roughly eight hours there.
mask = g != "Signer_UNK"
Xs, gs = X[mask], g[mask]
uniq = np.unique(gs)
sep_acc = sep_base = None

if len(uniq) < 2:
    print("SKIPPED: only %s present. This gate needs at least 2 signers." % uniq.tolist())
    print("Re-run once the full recording set is in place.")
else:
    # Split by SEQUENCE, not by frame. The 30 frames of one sequence are near
    # copies of each other, so a frame-level split puts near-duplicates on both
    # sides and lets the classifier recognise the sequence instead of the signer.
    # That inflates the exact number this gate exists to report.
    s_tr, s_te = train_test_split(np.arange(len(Xs)), test_size=0.3,
                                  stratify=gs, random_state=RANDOM_STATE)
    # Every 3rd frame only. Adjacent frames are near-copies, so they add fitting
    # time without adding information, and this gate is meant to be cheap enough
    # to run before committing the training machine.
    STRIDE = 3
    n_kept = len(range(0, Xs.shape[1], STRIDE))
    P_tr = Xs[s_tr][:, ::STRIDE].reshape(-1, DIM_COMPACT)   # pool frames, drop class
    P_te = Xs[s_te][:, ::STRIDE].reshape(-1, DIM_COMPACT)
    g_tr = np.repeat(gs[s_tr], n_kept)
    g_te = np.repeat(gs[s_te], n_kept)

    # Fitted once per arm prefix, because a bare "still 97%" verdict is a dead
    # end. Where the identity sits decides what to do about it: identity in
    # block A is hand geometry, which cannot be removed without removing the
    # signal with it, whereas identity concentrated in C or D is the per-signer
    # baseline that Next-Steps 4.6 keeps a fix in reserve for.
    _, bc = np.unique(g_te, return_counts=True)
    sep_base = float(bc.max() / bc.sum())
    sep_by_arm, clf, sc = {}, None, None

    for _name, _cut in ARM_CUTS.items():
        _sc = StandardScaler().fit(P_tr[:, :_cut])
        _clf = LogisticRegression(max_iter=1000, n_jobs=-1)
        _clf.fit(_sc.transform(P_tr[:, :_cut]), g_tr)
        sep_by_arm[_name] = float(_clf.score(_sc.transform(P_te[:, :_cut]), g_te))
        clf, sc = _clf, _sc              # the last arm is the full vector

    sep_acc = sep_by_arm[list(ARM_CUTS)[-1]]

    print("Signer recoverable from compact features: %.1f%%" % (sep_acc * 100))
    print("Majority-class baseline:                  %.1f%%" % (sep_base * 100))
    print("Phase 0, same check on 447-dim landmarks: 99.7%")
    print("(Phase 0 split by frame, this splits by sequence, so a few points of")
    print(" the drop are method rather than representation. The gap is far larger")
    print(" than that difference can account for either way.)")
    print()
    # This number bounds what may be CLAIMED, and does not on its own predict
    # LOSO accuracy. Round 1 is the counterexample: 99.7% separability sat
    # alongside hand-only scoring the best LOSO of any arm, 67.54%. Identity
    # being recoverable by a probe is not the same as the classifier relying on
    # it. So a high number here reframes the paper's claim, it does not cancel
    # the grid.
    if sep_acc > 0.95:
        print("  RED for the CLAIM, not for the run. Identity survives the")
        print("  redesign nearly intact, so Next-Steps 4.2 cannot be written as")
        print("  'the compact representation removes signer identity'. What it")
        print("  supports is the narrower, still-useful claim: a 3x smaller")
        print("  representation at equal or better signer-independent accuracy.")
        print("  Run the grid anyway. Round 1 paired 99.7% separability with the")
        print("  best LOSO arm, so this number does not predict the outcome.")
    elif sep_acc > 0.75:
        print("  AMBER. A real drop from 99.7%, but identity is still readable.")
        print("  Report the figure and keep the 4.2 claim quantitative rather")
        print("  than absolute: identity is reduced, not removed.")
    else:
        print("  GREEN. Identity is largely gone from the representation. This is")
        print("  the direct evidence for the Next-Steps 4.2 claim, and it belongs")
        print("  in the paper next to the ablation table.")

    print("\nSigner recoverable from each arm prefix (where the identity sits):")
    _prev = sep_base
    for _name, _cut in ARM_CUTS.items():
        _a = sep_by_arm[_name]
        print("  %-16s %3d dim   %5.1f%%   %+5.1f pp vs the arm above"
              % (_name, _cut, _a * 100, (_a - _prev) * 100))
        _prev = _a
    print("  (first row is measured against the %.1f%% majority baseline)"
          % (sep_base * 100))
    if sep_by_arm[list(ARM_CUTS)[0]] > 0.90:
        print("\n  Note: hands alone already identify the signer. Hand geometry is")
        print("  person-specific and cannot be normalised away without discarding")
        print("  the handshape signal, so this floor is not removable by feature")
        print("  design. It bounds how much any block-level change can achieve.")

    per = {s: float((clf.predict(sc.transform(P_te[g_te == s])) == s).mean())
           for s in uniq}
    print("\nPer-signer recall (who stays most identifiable):")
    for s, v in sorted(per.items(), key=lambda kv: -kv[1]):
        print("  %-22s %5.1f%%" % (s, v * 100))

In [ ]:
# ============================================================
# 6 -- Save
# ============================================================
np.savez_compressed(
    OUT_PATH,
    X=X, y=y, groups=g, filenames=fn,
    class_names=np.asarray(class_names),
    arm_names=np.asarray(list(ARM_CUTS)),
    arm_cuts=np.asarray(list(ARM_CUTS.values())),
    nms_focused=np.asarray([c for c, m in zip(class_names, nms_mask) if m]),
    nms_mask=nms_mask,
    mirror_perm=mirror_perm, mirror_sign=mirror_sign,
)
meta = {
    "n_sequences": int(len(X)),
    "n_classes": int(num_classes),
    "dim": int(DIM_COMPACT),
    "sequence_length": int(SEQUENCE_LENGTH),
    "max_samples_per_class": int(MAX_SAMPLES),
    "near_dup_rel_threshold": float(NEAR_DUP_REL_THRESHOLD),
    "n_removed_as_duplicate": int((~keep).sum()),
    "signers": {s: int((g == s).sum()) for s in np.unique(g)},
    "arm_cuts": ARM_CUTS,
    "block_d_features": BLOCK_D_FEATURES,
    "signer_separability": sep_acc,
    "signer_separability_by_arm": sep_by_arm if uniq.size > 1 else None,
    "signer_separability_baseline": sep_base,
    "empty_class_signer_cells": len(empty),
    "n_nms_focused_classes": int(nms_mask.sum()),
}
with open("dataset_meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)

print("\nWrote %s: X %s, %d classes, %d signers"
      % (OUT_PATH, X.shape, num_classes, len(np.unique(g))))
print("Wrote dataset_meta.json")
print("\nArm cuts for run_ablation:")
for n, c in ARM_CUTS.items():
    print("  %-16s X[..., :%d]" % (n, c))